# 02 传统监督学习对比

## 1. 传统机器学习概述

### 1.1 监督学习原理

监督学习是机器学习最核心的范式：给定带标签的训练数据 $(X, y)$，学习一个映射函数 $f: X \rightarrow y$，使得对未见过的样本也能准确预测。在裂缝识别任务中，$X$ 是从路面图像提取的特征向量，$y \in \{0, 1\}$ 表示无裂缝/有裂缝。

### 1.2 七种模型简介

本章对比七种经典监督学习模型，按算法原理分为六大范式：

| 范式 | 模型 | 核心思想 | 特点 |
|------|------|----------|------|
| **树模型** | 决策树 (DT) | 递归划分特征空间 | 可解释性强，易过拟合 |
| **核方法** | SVM | 寻找最大间隔超平面 | 高维特征有效，核函数灵活 |
| **概率方法** | 朴素贝叶斯 (NB) | 贝叶斯定理+特征独立性假设 | 训练极快，高维特征表现好 |
| **Bagging集成** | 随机森林 (RF) | 多棵树并行投票 | 降方差，抗过拟合 |
| **线性方法** | 逻辑回归 (LR) | 线性决策边界+概率输出 | 简单高效，可解释 |
| **Boosting集成** | XGBoost | 梯度提升树（串行优化） | 精度高，工业界主流 |
| **Boosting集成** | LightGBM | 直方图算法+Leaf-wise生长 | 训练快，内存省 |

### 1.3 本章结构

每种模型独立成节，覆盖以下环节：
1. **原理简介**：算法核心思想和数学基础
2. **模型实现**：基于 sklearn/xgboost/lightgbm 的封装代码
3. **参数搜索**：使用 GridSearchCV 搜索最优超参数
4. **性能评估**：混淆矩阵、ROC 曲线、五项指标
5. **参数影响分析**：关键参数对性能的影响曲线

七种模型统一使用 01 章确定的最优数据处理方案（CLAHE + 中值滤波，全部特征组合），以确保可比性。

## 2. 实验设计

### 2.1 统一数据管线

为保证模型间的可比性，所有模型使用相同的数据预处理管线（移植自 01 章的核心函数）。

### 2.2 评估指标

| 指标 | 公式 | 含义 |
|------|------|------|
| **准确率 (Accuracy)** | (TP+TN)/(TP+TN+FP+FN) | 整体分类正确率 |
| **精确率 (Precision)** | TP/(TP+FP) | 预测为正的样本中真正的正样本比例 |
| **召回率 (Recall)** | TP/(TP+FN) | 真正样本中被正确识别的比例 |
| **F1分数** | 2×P×R/(P+R) | 精确率和召回率的调和平均 |
| **ROC-AUC** | — | ROC曲线下面积，衡量模型区分能力 |

裂缝检测任务更关注召回率（漏检裂缝的代价高），同时需要精确率不会过低（误报过多影响可用性）。

In [ ]:
# ========================================
# 2.3 导入依赖库与配置
# ========================================
import os
from pathlib import Path
from typing import Tuple, Dict, Any
import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import torch

# 中文字体
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# 路径配置
from dotenv import load_dotenv
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")
_DATA_ROOT = os.getenv("CRACK_DATA_ROOT")
DATA_ROOT = Path(_DATA_ROOT).expanduser() if _DATA_ROOT else PROJECT_ROOT / "data"
if not DATA_ROOT.is_absolute():
    DATA_ROOT = PROJECT_ROOT / DATA_ROOT
DATA_ROOT = DATA_ROOT.resolve()
POSITIVE_DIR = DATA_ROOT / "Positive"
NEGATIVE_DIR = DATA_ROOT / "Negative"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

np.random.seed(42)
print(f"数据根目录: {DATA_ROOT}")
print(f"计算设备: {DEVICE}")

In [ ]:
# ========================================
# 2.4 数据处理与特征提取函数（移植自 01 章，自包含）
# ========================================

# ----- 图像读取 -----
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def _imread_gray(path: Path):
    buf = np.fromfile(str(path), dtype=np.uint8)
    if buf is None or buf.size == 0:
        return None
    return cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE)

def load_dataset(data_root: Path, max_samples: int | None = None):
    def _load_dir(directory, label):
        imgs, lbls = [], []
        for p in sorted(directory.iterdir())[:max_samples]:
            if p.suffix.lower() in IMAGE_EXTS:
                img = _imread_gray(p)
                if img is not None:
                    imgs.append(img); lbls.append(label)
        return imgs, lbls
    pos_imgs, pos_lbls = _load_dir(data_root / "Positive", 1)
    neg_imgs, neg_lbls = _load_dir(data_root / "Negative", 0)
    all_imgs = pos_imgs + neg_imgs
    labels = np.array(pos_lbls + neg_lbls, dtype=np.int64)
    shapes = {img.shape for img in all_imgs}
    if len(shapes) == 1:
        images = np.stack(all_imgs)
    else:
        images = np.array(all_imgs, dtype=object)
    return images, labels

# ----- 预处理 -----
def apply_clahe(img, clip_limit=2.0, tile_grid_size=(8,8)):
    return cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size).apply(img)

def apply_gaussian_filter(img, kernel_size=(5,5), sigma=1.0):
    return cv2.GaussianBlur(img, kernel_size, sigma)

def apply_median_filter(img, kernel_size=5):
    return cv2.medianBlur(img, kernel_size)

def default_preprocess(img):
    """默认预处理管线：CLAHE + 中值滤波"""
    return apply_median_filter(apply_clahe(img))

# ----- 特征提取 -----
def extract_hog_features(img, orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2)):
    return hog(img, orientations=orientations, pixels_per_cell=pixels_per_cell,
               cells_per_block=cells_per_block, feature_vector=True)

def extract_lbp_features(img, radius=1, n_points=8):
    n_bins = n_points * (n_points - 1) + 3
    lbp_img = local_binary_pattern(img, n_points, radius, method="uniform")
    hist, _ = np.histogram(lbp_img, bins=n_bins, range=(0, n_bins), density=True)
    return hist

def extract_glcm_features(img, distances=(1,3,5), angles=(0, np.pi/4, np.pi/2, 3*np.pi/4)):
    img_u8 = img.astype(np.uint8) if img.dtype != np.uint8 else img
    props = []
    for d in distances:
        for a in angles:
            glcm = graycomatrix(img_u8, distances=[d], angles=[a],
                                levels=256, symmetric=True, normed=True)
            props.extend([graycoprops(glcm, "contrast")[0,0],
                          graycoprops(glcm, "correlation")[0,0],
                          graycoprops(glcm, "energy")[0,0],
                          graycoprops(glcm, "homogeneity")[0,0]])
    return np.array(props, dtype=np.float64)

def extract_edge_density(img, low_threshold=50, high_threshold=150):
    edges = cv2.Canny(img, low_threshold, high_threshold)
    return float(np.count_nonzero(edges)) / edges.size

def extract_all_features(image: np.ndarray) -> np.ndarray:
    """提取全部特征并拼接（HOG+LBP+GLCM+边缘密度）。"""
    image = image.astype(np.uint8) if image.dtype != np.uint8 else image
    return np.concatenate([
        extract_hog_features(image),
        extract_lbp_features(image),
        extract_glcm_features(image),
        np.array([extract_edge_density(image)], dtype=np.float64),
    ]).astype(np.float64)

# ----- 数据划分 -----
def split_dataset(images, labels, train_ratio=0.7, val_ratio=0.0, random_seed=42):
    val_test_ratio = 1.0 - train_ratio
    X_train, X_temp, y_train, y_temp = train_test_split(
        images, labels, test_size=val_test_ratio,
        random_state=random_seed, stratify=labels)
    if val_ratio == 0.0:
        return X_train, None, X_temp, y_train, None, y_temp
    test_ratio_in_temp = (1.0 - train_ratio - val_ratio) / (1.0 - train_ratio)
    try:
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=test_ratio_in_temp,
            random_state=random_seed, stratify=y_temp)
    except ValueError:
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=test_ratio_in_temp, random_state=random_seed)
    return X_train, X_val, X_test, y_train, y_val, y_test

print("数据处理和特征提取函数定义完成。（自包含，无需依赖外部模块）")

In [ ]:
# ========================================
# 2.5 加载数据并提取特征
# ========================================

# 快速运行模式：仅加载部分样本；完整实验设为 False
QUICK_RUN = True
max_samples = 1000 if QUICK_RUN else None

print("正在加载数据集...")
images, labels = load_dataset(DATA_ROOT, max_samples=max_samples)
print(f"加载完成: {len(labels)} 张图像 "
      f"(正={int((labels==1).sum())}, 负={int((labels==0).sum())})")

# 预处理并提取特征
print("\n正在预处理并提取特征（CLAHE + 中值滤波 + 全部特征）...")
processed_images = np.array([default_preprocess(img) for img in images])
X_features = np.stack([extract_all_features(img) for img in processed_images])
y = labels
print(f"特征矩阵形状: {X_features.shape}")
print(f"特征维度: {X_features.shape[1]}")

# 划分为训练集和测试集（留出法 70/30 + 分层抽样）
X_train, _, X_test, y_train, _, y_test = split_dataset(
    X_features, y, train_ratio=0.7, val_ratio=0.0, random_seed=42
)
print(f"\n数据集划分: 训练集={len(y_train)} 样本, 测试集={len(y_test)} 样本")
print(f"训练集类别分布: 正={int((y_train==1).sum())}, 负={int((y_train==0).sum())}")

## 3. 评估工具函数

定义统一的模型评估函数，所有七种模型共享以下评估管线：
- `evaluate_model()` — 计算五项指标 + 绘制混淆矩阵和ROC曲线
- `plot_model_comparison()` — 多模型对比可视化

In [ ]:
# ========================================
# 3.1 统一评估函数（移植自 src/evaluation/metrics.py）
# ========================================

def evaluate_model(model, X_tr, y_tr, X_te, y_te, model_name="模型",
                   class_names=("无裂缝", "有裂缝")):
    """训练并全面评估一个模型。

    Returns
    -------
    dict : 包含 metrics (dict), confusion_matrix_fig, roc_fig
    """
    # 训练
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    # 概率预测（用于 ROC-AUC）
    try:
        y_prob = model.predict_proba(X_te)[:, 1]
        roc_auc = roc_auc_score(y_te, y_prob)
    except (AttributeError, RuntimeError):
        y_prob = None
        roc_auc = float("nan")

    metrics = {
        "模型": model_name,
        "准确率": accuracy_score(y_te, y_pred),
        "精确率": precision_score(y_te, y_pred, zero_division=0),
        "召回率": recall_score(y_te, y_pred, zero_division=0),
        "F1分数": f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC": roc_auc,
    }

    # 混淆矩阵
    cm = confusion_matrix(y_te, y_pred, labels=[0, 1])
    fig_cm, ax_cm = plt.subplots(figsize=(5, 4))
    im = ax_cm.imshow(cm, cmap="Blues")
    ax_cm.set_xticks([0, 1]); ax_cm.set_yticks([0, 1])
    ax_cm.set_xticklabels([f"预测 {class_names[0]}", f"预测 {class_names[1]}"])
    ax_cm.set_yticklabels([f"实际 {class_names[0]}", f"实际 {class_names[1]}"])
    for i in range(2):
        for j in range(2):
            ax_cm.text(j, i, str(cm[i,j]), ha="center", va="center", fontsize=14,
                      fontweight="bold", color="white" if cm[i,j] > cm.max()/2 else "black")
    ax_cm.set_title(f"{model_name} - 混淆矩阵\n(准确率: {metrics['准确率']:.4f})", fontsize=12)
    plt.colorbar(im, ax=ax_cm)
    plt.tight_layout()

    # ROC 曲线
    fig_roc = None
    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_te, y_prob)
        fig_roc, ax_roc = plt.subplots(figsize=(5, 4))
        ax_roc.plot(fpr, tpr, color="tomato", linewidth=2,
                    label=f"{model_name} (AUC={roc_auc:.4f})")
        ax_roc.plot([0,1], [0,1], "gray", linestyle="--", alpha=0.5, label="随机猜测")
        ax_roc.set_xlabel("假阳性率 (FPR)"); ax_roc.set_ylabel("真阳性率 (TPR)")
        ax_roc.set_title(f"{model_name} - ROC 曲线"); ax_roc.legend(loc="lower right")
        ax_roc.grid(True, alpha=0.3)
        plt.tight_layout()

    plt.show()
    return {"metrics": metrics, "cm_fig": fig_cm, "roc_fig": fig_roc}


def cross_validate_model(model_factory, X, y, cv_folds=5, random_seed=42):
    """K折分层交叉验证评估模型稳定性。"""
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_seed)
    fold_results = []
    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), 1):
        model = model_factory()
        model.fit(X[tr_idx], y[tr_idx])
        y_pred = model.predict(X[te_idx])
        try:
            y_prob = model.predict_proba(X[te_idx])[:, 1]
            roc_auc = roc_auc_score(y[te_idx], y_prob)
        except Exception:
            roc_auc = float("nan")
        fold_results.append({
            "fold": fold,
            "准确率": accuracy_score(y[te_idx], y_pred),
            "精确率": precision_score(y[te_idx], y_pred, zero_division=0),
            "召回率": recall_score(y[te_idx], y_pred, zero_division=0),
            "F1分数": f1_score(y[te_idx], y_pred, zero_division=0),
            "ROC-AUC": roc_auc,
        })
    df = pd.DataFrame(fold_results)
    means = {c: df[c].mean() for c in df.columns if c != "fold"}
    stds = {c: df[c].std() for c in df.columns if c != "fold"}
    df = pd.concat([df, pd.DataFrame([{"fold": "均值", **means}, {"fold": "标准差", **stds}])],
                   ignore_index=True)
    return df

print("评估工具函数定义完成。")

In [ ]:
# ========================================
# 3.2 多模型对比可视化函数
# ========================================

def plot_model_comparison(results_df: pd.DataFrame, title="模型性能对比"):
    """绘制多模型性能对比图（并排柱状图 + ROC曲线叠加）。"""
    df = results_df.dropna(subset=["F1分数"])
    metrics_cols = ["准确率", "精确率", "召回率", "F1分数"]

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(df))
    width = 0.2
    colors = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12"]

    for i, (col, color) in enumerate(zip(metrics_cols, colors)):
        vals = [float(str(v).split("±")[0]) if isinstance(v, str) else float(v)
                for v in df[col]]
        bars = ax.bar(x + i * width, vals, width, label=col, color=color, alpha=0.85)

    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(df["模型"], rotation=30, ha="right", fontsize=10)
    ax.set_ylabel("分数", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.legend(loc="lower right")
    ax.set_ylim(0.5, 1.0)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

print("多模型对比可视化函数定义完成。")

## 4. 决策树（DT）

### 原理

决策树通过递归地将特征空间划分为多个区域，每个叶节点对应一个类别预测。分裂标准通常使用基尼不纯度（Gini）或信息增益（Entropy）。

**优点**：可解释性强，无需特征缩放，能捕捉非线性关系。
**缺点**：容易过拟合，对噪声敏感，泛化能力较弱。

### 参数搜索空间

| 参数 | 搜索范围 | 含义 |
|------|----------|------|
| max_depth | [3,5,10,15,20,None] | 树的最大深度 |
| criterion | ["gini","entropy"] | 分裂标准 |
| min_samples_split | [2,5,10] | 内部节点最小样本数 |

In [ ]:
# ========================================
# 4.1 决策树模型定义、参数搜索与评估
# ========================================
import time

# ----- 参数搜索 -----
print(">>> 决策树 参数搜索（GridSearchCV）...")
if "DT" == "RF":
    param_grid = {
        'n_estimators': [50, 100, 200, 500],
        'max_depth': [5, 10, 20, None],
        'min_samples_split': [2, 5, 10],
    }
    base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
elif "DT" == "DT":
    param_grid = {
        'max_depth': [3, 5, 10, 15, 20, None],
        'criterion': ['gini', 'entropy'],
        'min_samples_split': [2, 5, 10],
    }
    base_model = DecisionTreeClassifier(random_state=42)
elif "DT" == "LR":
    param_grid = {
        'logreg__C': [0.01, 0.1, 1, 10, 100],
        'logreg__penalty': ['l1', 'l2'],
        'logreg__solver': ['liblinear', 'lbfgs'],
    }
    base_model = Pipeline([
        ('scaler', StandardScaler()),
        ('logreg', LogisticRegression(random_state=42, max_iter=1000))
    ])

t0 = time.time()
grid = GridSearchCV(base_model, param_grid, cv=3, scoring='f1',
                    n_jobs=-1, verbose=0)
grid.fit(X_train, y_train)
elapsed = time.time() - t0

print(f"搜索完成 (耗时: {elapsed:.1f}s)")
print(f"最优参数: {grid.best_params_}")
print(f"最优F1分数 (CV): {grid.best_score_:.4f}")

# ----- 评估 -----
best_model = grid.best_estimator_
result = evaluate_model(best_model, X_train, y_train, X_test, y_test,
                        model_name="决策树 (最优)")
print("\n测试集评估指标：")
for k, v in result["metrics"].items():
    if k != "模型":
        print(f"  {k}: {v:.4f}")

# ----- 交叉验证 -----
print("\n>>> 5折交叉验证稳定性评估...")
cv_df = cross_validate_model(lambda: DecisionTreeClassifier(**grid.best_params_, random_state=42),
    X_features, y, cv_folds=5)
display(cv_df.style.format({c: "{:.4f}" for c in cv_df.columns if c != "fold"}))

In [ ]:
# ========================================
# 4.2 关键参数影响分析：max_depth 对性能的影响
# ========================================

depths = [3, 5, 10, 15, 20, 30, None]
depths_labels = [str(d) for d in depths]
train_scores, test_scores = [], []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_scores.append(f1_score(y_train, dt.predict(X_train)))
    test_scores.append(f1_score(y_test, dt.predict(X_test)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(depths_labels, train_scores, 'o-', color='#3498db', linewidth=2, label='训练集 F1')
ax.plot(depths_labels, test_scores, 's-', color='#e74c3c', linewidth=2, label='测试集 F1')
ax.set_xlabel('最大深度 (max_depth)', fontsize=12)
ax.set_ylabel('F1分数', fontsize=12)
ax.set_title('决策树：max_depth 对性能的影响', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("从图中可以看出，深度过小时欠拟合（训练和测试分数都低），"
      "深度过大时过拟合（训练分数高但测试分数下降或持平）。")

## 5. SVM（SVM）

### 原理

支持向量机通过寻找最大化类别间隔的超平面进行分类。对于非线性可分数据，通过核函数（kernel function）将数据映射到高维空间，在高维空间中寻找线性超平面。

**优点**：高维特征空间有效，核函数灵活（linear/rbf/poly），泛化能力强。
**缺点**：大数据集训练慢，需要特征缩放，核函数和参数选择敏感。

### 参数搜索空间

| 参数 | 搜索范围 | 含义 |
|------|----------|------|
| kernel | ["linear","rbf","poly"] | 核函数类型 |
| C | [0.1,1,10,100] | 正则化参数（越大越易过拟合） |
| gamma | ["scale","auto",0.01,0.1] | 核函数系数（仅rbf/poly）|

In [ ]:
# ========================================
# 5.1 SVM模型定义、参数搜索与评估
# ========================================
import time

# ----- 参数搜索（缩减版，SVM在大数据集上慢） -----
print(">>> SVM 参数搜索（GridSearchCV）...")
param_grid = {
    'svc__kernel': ['linear', 'rbf', 'poly'],
    'svc__C': [0.1, 1, 10],
    'svc__gamma': ['scale', 'auto'],
}

pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(probability=True, random_state=42))
])
grid = GridSearchCV(pipe_svm, param_grid, cv=3, scoring='f1',
                    n_jobs=-1, verbose=0)
t0 = time.time()
grid.fit(X_train, y_train)
elapsed = time.time() - t0

print(f"搜索完成 (耗时: {elapsed:.1f}s)")
print(f"最优参数: {grid.best_params_}")
print(f"最优F1分数 (CV): {grid.best_score_:.4f}")

# ----- 评估 -----
best_model = grid.best_estimator_
result = evaluate_model(best_model, X_train, y_train, X_test, y_test,
                        model_name="SVM (最优)")
print("\n测试集评估指标：")
for k, v in result["metrics"].items():
    if k != "模型":
        print(f"  {k}: {v:.4f}")

# ----- 交叉验证 -----
print("\n>>> 5折交叉验证稳定性评估...")
def make_svm():
    return Pipeline([('scaler', StandardScaler()),
        ('svc', SVC(probability=True, random_state=42, **{k.split('__')[1]: v
            for k, v in grid.best_params_.items()}))])
cv_df = cross_validate_model(make_svm, X_features, y, cv_folds=5)
display(cv_df.style.format({c: "{:.4f}" for c in cv_df.columns if c != "fold"}))

In [ ]:
# ========================================
# 5.2 关键参数影响分析：kernel 和 C 对性能的影响
# ========================================

kernels = ['linear', 'rbf', 'poly']
C_values = [0.1, 1, 10, 100]
results_kernel = []
for k in kernels:
    for c in C_values:
        svm = Pipeline([('scaler', StandardScaler()),
                        ('svc', SVC(kernel=k, C=c, random_state=42))])
        svm.fit(X_train, y_train)
        results_kernel.append({
            'kernel': k, 'C': c,
            'F1': f1_score(y_test, svm.predict(X_test)),
        })
df_svm = pd.DataFrame(results_kernel)

fig, ax = plt.subplots(figsize=(8, 5))
for k in kernels:
    subset = df_svm[df_svm['kernel'] == k]
    ax.plot(subset['C'].astype(str), subset['F1'], 'o-', linewidth=2, label=f'kernel={k}')
ax.set_xlabel('C (正则化参数)', fontsize=12)
ax.set_ylabel('F1分数', fontsize=12)
ax.set_title('SVM：核函数和正则化参数对性能的影响', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("rbf核通常在非线性数据上表现最好；C 值需要平衡间隔最大化与分类误差。")

## 6. 朴素贝叶斯（NB）

### 原理

朴素贝叶斯基于贝叶斯定理，在"特征之间相互独立"的朴素假设下，计算每个类别的后验概率 $P(y|X) \propto P(y)\prod_i P(x_i|y)$，选择后验概率最大的类别。

对于连续特征，使用高斯朴素贝叶斯（GaussianNB），假设每个特征在每个类别下服从高斯分布。

**优点**：训练速度极快，高维特征表现好，对小数据集友好，天然输出概率。
**缺点**：特征独立性假设在实际中不总是成立，可能影响精度。

### 参数搜索空间

| 参数 | 搜索范围 | 含义 |
|------|----------|------|
| var_smoothing | [1e-9, 1e-7, 1e-5, 1e-3] | 方差平滑参数（防止零方差） |

In [ ]:
# ========================================
# 6.1 朴素贝叶斯模型定义、参数搜索与评估
# ========================================
import time

# ----- 参数搜索 -----
print(">>> 朴素贝叶斯 参数搜索（GridSearchCV）...")
# 尝试 GaussianNB 和 MultinomialNB
param_grid = {
    'var_smoothing': [1e-9, 1e-7, 1e-5, 1e-3]
}

pipe_nb = Pipeline([
    ('scaler', StandardScaler()),
    ('nb', GaussianNB())
])
grid = GridSearchCV(pipe_nb, {'nb__var_smoothing': [1e-9, 1e-7, 1e-5, 1e-3]},
                    cv=3, scoring='f1', n_jobs=-1, verbose=0)
t0 = time.time()
grid.fit(X_train, y_train)
elapsed = time.time() - t0

print(f"搜索完成 (耗时: {elapsed:.1f}s)")
print(f"最优参数: {grid.best_params_}")
print(f"最优F1分数 (CV): {grid.best_score_:.4f}")

# ----- 评估 -----
best_model = grid.best_estimator_
result = evaluate_model(best_model, X_train, y_train, X_test, y_test,
                        model_name="朴素贝叶斯 (最优)")
print("\n测试集评估指标：")
for k, v in result["metrics"].items():
    if k != "模型":
        print(f"  {k}: {v:.4f}")

# ----- 交叉验证 -----
print("\n>>> 5折交叉验证稳定性评估...")
def make_nb():
    return Pipeline([('scaler', StandardScaler()),
                     ('nb', GaussianNB(var_smoothing=grid.best_params_['nb__var_smoothing']))])
cv_df = cross_validate_model(make_nb, X_features, y, cv_folds=5)
display(cv_df.style.format({c: "{:.4f}" for c in cv_df.columns if c != "fold"}))

In [ ]:
# ========================================
# 6.2 关键参数影响分析：var_smoothing 对性能的影响
# ========================================

smoothings = [1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
test_f1_scores = []
for vs in smoothings:
    nb = Pipeline([('scaler', StandardScaler()),
                   ('nb', GaussianNB(var_smoothing=vs))])
    nb.fit(X_train, y_train)
    test_f1_scores.append(f1_score(y_test, nb.predict(X_test)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(smoothings, test_f1_scores, 'o-', color='#2ecc71', linewidth=2)
ax.set_xlabel('方差平滑参数 (对数尺度)', fontsize=12)
ax.set_ylabel('测试集 F1分数', fontsize=12)
ax.set_title('朴素贝叶斯：var_smoothing 对性能的影响', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("var_smoothing 控制方差平滑程度，影响数值稳定性和泛化能力。")

## 7. 随机森林（RF）

### 原理

随机森林是 Bagging（Bootstrap Aggregating）集成方法的代表。通过 Bootstrap 采样生成多个不同的训练子集，每个子集训练一棵决策树，最终通过多数投票决定分类结果。每棵树在分裂时只考虑随机选取的特征子集，进一步增加树之间的多样性。

**优点**：抗过拟合，对高维特征稳健，可评估特征重要性，训练可并行。
**缺点**：模型体积大，推理速度慢于单棵树，解释性不如决策树。

### 参数搜索空间

| 参数 | 搜索范围 | 含义 |
|------|----------|------|
| n_estimators | [50,100,200,500] | 树的数量 |
| max_depth | [5,10,20,None] | 最大深度 |
| min_samples_split | [2,5,10] | 内部节点最小样本数 |

In [ ]:
# ========================================
# 7.1 随机森林模型定义、参数搜索与评估
# ========================================
import time

# ----- 参数搜索 -----
print(">>> 随机森林 参数搜索（GridSearchCV）...")
if "RF" == "RF":
    param_grid = {
        'n_estimators': [50, 100, 200, 500],
        'max_depth': [5, 10, 20, None],
        'min_samples_split': [2, 5, 10],
    }
    base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
elif "RF" == "DT":
    param_grid = {
        'max_depth': [3, 5, 10, 15, 20, None],
        'criterion': ['gini', 'entropy'],
        'min_samples_split': [2, 5, 10],
    }
    base_model = DecisionTreeClassifier(random_state=42)
elif "RF" == "LR":
    param_grid = {
        'logreg__C': [0.01, 0.1, 1, 10, 100],
        'logreg__penalty': ['l1', 'l2'],
        'logreg__solver': ['liblinear', 'lbfgs'],
    }
    base_model = Pipeline([
        ('scaler', StandardScaler()),
        ('logreg', LogisticRegression(random_state=42, max_iter=1000))
    ])

t0 = time.time()
grid = GridSearchCV(base_model, param_grid, cv=3, scoring='f1',
                    n_jobs=-1, verbose=0)
grid.fit(X_train, y_train)
elapsed = time.time() - t0

print(f"搜索完成 (耗时: {elapsed:.1f}s)")
print(f"最优参数: {grid.best_params_}")
print(f"最优F1分数 (CV): {grid.best_score_:.4f}")

# ----- 评估 -----
best_model = grid.best_estimator_
result = evaluate_model(best_model, X_train, y_train, X_test, y_test,
                        model_name="随机森林 (最优)")
print("\n测试集评估指标：")
for k, v in result["metrics"].items():
    if k != "模型":
        print(f"  {k}: {v:.4f}")

# ----- 交叉验证 -----
print("\n>>> 5折交叉验证稳定性评估...")
cv_df = cross_validate_model(lambda: RandomForestClassifier(**grid.best_params_, random_state=42),
    X_features, y, cv_folds=5)
display(cv_df.style.format({c: "{:.4f}" for c in cv_df.columns if c != "fold"}))

## 8. 逻辑回归（LR）

### 原理

逻辑回归使用 Sigmoid 函数 $\sigma(z) = 1/(1+e^{-z})$ 将线性组合 $z = w^Tx + b$ 映射到 (0,1) 区间，作为正类的概率估计。通过最大化对数似然（等价于最小化交叉熵损失）来训练。

**优点**：简单高效，训练极快，输出校准良好的概率，可解释性强（权重直接反映特征重要性）。
**缺点**：只能学习线性决策边界，对特征缩放敏感，对异常值敏感。

### 参数搜索空间

| 参数 | 搜索范围 | 含义 |
|------|----------|------|
| C | [0.01,0.1,1,10,100] | 正则化强度的倒数 |
| penalty | ["l1","l2"] | 正则化类型 |
| solver | ["liblinear","lbfgs"] | 优化算法 |

In [ ]:
# ========================================
# 8.1 逻辑回归模型定义、参数搜索与评估
# ========================================
import time

# ----- 参数搜索 -----
print(">>> 逻辑回归 参数搜索（GridSearchCV）...")
param_grid = {
    'logreg__C': [0.01, 0.1, 1, 10, 100],
    'logreg__penalty': ['l1', 'l2'],
    'logreg__solver': ['liblinear', 'lbfgs'],
}
base_model = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(random_state=42, max_iter=1000))
])

t0 = time.time()
grid = GridSearchCV(base_model, param_grid, cv=3, scoring='f1',
                    n_jobs=-1, verbose=0)
grid.fit(X_train, y_train)
elapsed = time.time() - t0

print(f"搜索完成 (耗时: {elapsed:.1f}s)")
print(f"最优参数: {grid.best_params_}")
print(f"最优F1分数 (CV): {grid.best_score_:.4f}")

# ----- 评估 -----
best_model = grid.best_estimator_
result = evaluate_model(best_model, X_train, y_train, X_test, y_test,
                        model_name="逻辑回归 (最优)")
print("\n测试集评估指标：")
for k, v in result["metrics"].items():
    if k != "模型":
        print(f"  {k}: {v:.4f}")

# ----- 交叉验证 -----
print("\n>>> 5折交叉验证稳定性评估...")
from sklearn.base import clone
cv_df = cross_validate_model(lambda: clone(grid.best_estimator_),
    X_features, y, cv_folds=5)
display(cv_df.style.format({c: "{:.4f}" for c in cv_df.columns if c != "fold"}))

## 9. XGBoost（XGBoost）

### 原理

XGBoost（eXtreme Gradient Boosting）是 Gradient Boosting 的高效实现。通过串行训练多棵决策树，每棵新树拟合前序树集合的残差（负梯度方向），逐步降低整体偏差。XGBoost 引入二阶泰勒展开近似损失函数、正则化项控制复杂度、列采样等优化。

**优点**：精度极高（Kaggle 竞赛常胜算法），内置正则化防过拟合，支持自定义损失函数。
**缺点**：参数多调参复杂，训练时间较长，对异常值敏感。

### 参数搜索空间

| 参数 | 搜索范围 | 含义 |
|------|----------|------|
| n_estimators | [50,100,200] | 树的数量 |
| max_depth | [3,6,9] | 树的最大深度 |
| learning_rate | [0.01,0.1,0.3] | 学习率（收缩步长） |
| subsample | [0.8,1.0] | 训练每棵树时的样本采样比例 |

In [ ]:
# ========================================
# 9.1 XGBoost模型定义、参数搜索与评估
# ========================================
import time

# 尝试导入XGBoost
try:
    from xgboost import XGBClassifier
    print("XGBoost 导入成功。")
except ImportError:
    print("XGBoost 未安装，请运行: pip install xgboost")
    raise

# ----- 参数搜索 -----
print("\n>>> XGBoost 参数搜索（GridSearchCV）...")
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
}

base_model = XGBClassifier(random_state=42, n_jobs=-1, verbosity=0)
grid = GridSearchCV(base_model, param_grid, cv=3, scoring='f1',
                    n_jobs=-1, verbose=0)
t0 = time.time()
grid.fit(X_train, y_train)
elapsed = time.time() - t0

print(f"搜索完成 (耗时: {elapsed:.1f}s)")
print(f"最优参数: {grid.best_params_}")
print(f"最优F1分数 (CV): {grid.best_score_:.4f}")

# ----- 使用最优参数评估 -----
best_model = grid.best_estimator_
result = evaluate_model(best_model, X_train, y_train, X_test, y_test,
                        model_name="XGBoost (最优)")
print("\n测试集评估指标：")
for k, v in result["metrics"].items():
    if k != "模型":
        print(f"  {k}: {v:.4f}")

# ----- 交叉验证稳定性评估 -----
print("\n>>> 5折交叉验证稳定性评估...")
from sklearn.base import clone
cv_df = cross_validate_model(lambda: clone(grid.best_estimator_),
    X_features, y, cv_folds=5)
display(cv_df.style.format({c: "{:.4f}" for c in cv_df.columns if c != "fold"}))

## 10. LightGBM（LightGBM）

### 原理

LightGBM（Light Gradient Boosting Machine）是微软开源的 Gradient Boosting 框架。与 XGBoost 的核心差异在于：使用直方图（Histogram）算法将连续特征离散化以加速训练；采用 Leaf-wise（按叶生长）策略而非 Level-wise（按层生长），以更快降低损失；引入 Gradient-based One-Side Sampling (GOSS) 和 Exclusive Feature Bundling (EFB) 技术。

**优点**：训练速度比 XGBoost 快 3-10 倍，内存占用更低，处理大规模数据效率高。
**缺点**：小数据集可能过拟合（Leaf-wise 策略），参数调优同样复杂。

### 参数搜索空间

| 参数 | 搜索范围 | 含义 |
|------|----------|------|
| n_estimators | [50,100,200] | 树的数量 |
| max_depth | [3,6,9,-1] | 树的最大深度（-1为不限制） |
| learning_rate | [0.01,0.1,0.3] | 学习率 |
| num_leaves | [31,63,127] | 叶节点数（控制树复杂度） |

In [ ]:
# ========================================
# 10.1 LightGBM模型定义、参数搜索与评估
# ========================================
import time

# 尝试导入LightGBM
try:
    from lightgbm import LGBMClassifier
    print("LightGBM 导入成功。")
except ImportError:
    print("LightGBM 未安装，请运行: pip install lightgbm")
    raise

# ----- 参数搜索 -----
print("\n>>> LightGBM 参数搜索（GridSearchCV）...")
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 6, 9],
    'num_leaves': [31, 63, 127],
    'learning_rate': [0.01, 0.1, 0.3],
}

base_model = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
grid = GridSearchCV(base_model, param_grid, cv=3, scoring='f1',
                    n_jobs=-1, verbose=0)
t0 = time.time()
grid.fit(X_train, y_train)
elapsed = time.time() - t0

print(f"搜索完成 (耗时: {elapsed:.1f}s)")
print(f"最优参数: {grid.best_params_}")
print(f"最优F1分数 (CV): {grid.best_score_:.4f}")

# ----- 使用最优参数评估 -----
best_model = grid.best_estimator_
result = evaluate_model(best_model, X_train, y_train, X_test, y_test,
                        model_name="LightGBM (最优)")
print("\n测试集评估指标：")
for k, v in result["metrics"].items():
    if k != "模型":
        print(f"  {k}: {v:.4f}")

# ----- 交叉验证稳定性评估 -----
print("\n>>> 5折交叉验证稳定性评估...")
from sklearn.base import clone
cv_df = cross_validate_model(lambda: clone(grid.best_estimator_),
    X_features, y, cv_folds=5)
display(cv_df.style.format({c: "{:.4f}" for c in cv_df.columns if c != "fold"}))

## 11. 七模型综合对比

### 11.1 汇总评估

将前述七种模型的最优配置结果汇总对比，并从性能、速度、复杂度三个维度综合评价。

### 11.2 ROC 曲线叠加

将所有模型的 ROC 曲线绘制在同一张图上，直观比较不同模型的分类能力。

In [ ]:
# ========================================
# 11.1 七模型综合对比：训练、评估与汇总
# ========================================
import time

# --- 定义所有模型工厂函数（使用各模型搜索到的最优参数） ---
# 注意：这里使用简化参数以加速，实际应使用 GridSearchCV 的最优结果

model_factories = {
    "决策树 (DT)": lambda: DecisionTreeClassifier(max_depth=10, random_state=42),
    "SVM": lambda: Pipeline([('scaler', StandardScaler()),
                             ('svc', SVC(kernel='rbf', C=1.0, probability=True, random_state=42))]),
    "朴素贝叶斯 (NB)": lambda: Pipeline([('scaler', StandardScaler()),
                                          ('nb', GaussianNB(var_smoothing=1e-7))]),
    "随机森林 (RF)": lambda: RandomForestClassifier(n_estimators=100, max_depth=20,
                                                      random_state=42, n_jobs=-1),
    "逻辑回归 (LR)": lambda: Pipeline([('scaler', StandardScaler()),
                                        ('lr', LogisticRegression(C=1.0, random_state=42, max_iter=1000))]),
}

# 尝试导入 XGBoost 和 LightGBM
try:
    from xgboost import XGBClassifier
    model_factories["XGBoost"] = lambda: XGBClassifier(n_estimators=100, max_depth=6,
                                                        learning_rate=0.1, random_state=42,
                                                        n_jobs=-1, verbosity=0)
except ImportError:
    print("XGBoost 未安装，跳过。")

try:
    from lightgbm import LGBMClassifier
    model_factories["LightGBM"] = lambda: LGBMClassifier(n_estimators=100, max_depth=6,
                                                          learning_rate=0.1, random_state=42,
                                                          n_jobs=-1, verbose=-1)
except ImportError:
    print("LightGBM 未安装，跳过。")

# --- 训练并评估所有模型 ---
all_results = []
all_models_for_cv = {}

for name, factory in model_factories.items():
    print(f"\n{'='*50}")
    print(f"训练并评估: {name}")
    print(f"{'='*50}")
    t0 = time.time()
    model = factory()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_prob)
    except Exception:
        y_prob = None
        roc_auc = float("nan")

    t0 = time.time()
    model.predict(X_test)
    infer_time = (time.time() - t0) / len(y_test)

    all_results.append({
        "模型": name,
        "准确率": accuracy_score(y_test, y_pred),
        "精确率": precision_score(y_test, y_pred, zero_division=0),
        "召回率": recall_score(y_test, y_pred, zero_division=0),
        "F1分数": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc,
        "训练时间(s)": round(train_time, 2),
        "推理时间(ms/样本)": round(infer_time * 1000, 4),
    })
    all_models_for_cv[name] = (model, y_prob)
    print(f"  F1={all_results[-1]['F1分数']:.4f}, "
          f"训练={train_time:.1f}s")

df_all = pd.DataFrame(all_results).sort_values("F1分数", ascending=False)
print("\n" + "="*60)
print("七模型综合对比结果：")
print("="*60)
display(df_all.style
       .background_gradient(subset=["F1分数", "准确率"], cmap="Blues")
       .format({c: "{:.4f}" for c in ["准确率","精确率","召回率","F1分数","ROC-AUC"]}))

In [ ]:
# ========================================
# 11.2 综合对比可视化
# ========================================

# 左图：性能指标对比
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

metrics = ["准确率", "精确率", "召回率", "F1分数"]
x = np.arange(len(df_all))
width = 0.2
colors = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12"]
for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax1.bar(x + i*width, df_all[metric], width, label=metric, color=color, alpha=0.85)
ax1.set_xticks(x + width*1.5)
ax1.set_xticklabels(df_all["模型"], rotation=30, ha="right", fontsize=9)
ax1.set_ylabel("分数", fontsize=12)
ax1.set_title("七模型性能指标对比", fontsize=14, fontweight='bold')
ax1.legend(loc="lower right")
ax1.set_ylim(0.5, 1.0)
ax1.grid(axis="y", alpha=0.3)

# 右图：训练时间 vs F1分数
ax2.scatter(df_all["训练时间(s)"], df_all["F1分数"], s=200, c=range(len(df_all)),
           cmap="viridis", edgecolors="black", linewidth=1.5, zorder=5)
for _, row in df_all.iterrows():
    ax2.annotate(row["模型"].split("(")[0].strip(),
                (row["训练时间(s)"], row["F1分数"]),
                textcoords="offset points", xytext=(10, 5), fontsize=9)
ax2.set_xlabel("训练时间 (秒)", fontsize=12)
ax2.set_ylabel("F1分数", fontsize=12)
ax2.set_title("模型性能 vs 训练效率", fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ROC 曲线叠加
fig_roc, ax_roc = plt.subplots(figsize=(8, 7))
colors_roc = plt.cm.tab10(np.linspace(0, 1, len(all_models_for_cv)))
for (name, (model, y_prob)), color in zip(all_models_for_cv.items(), colors_roc):
    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        ax_roc.plot(fpr, tpr, linewidth=2, color=color, label=f"{name} (AUC={auc:.4f})")
ax_roc.plot([0,1], [0,1], "gray", linestyle="--", alpha=0.5, label="随机猜测")
ax_roc.set_xlabel("假阳性率 (FPR)", fontsize=12)
ax_roc.set_ylabel("真阳性率 (TPR)", fontsize=12)
ax_roc.set_title("七模型 ROC 曲线叠加对比", fontsize=14, fontweight='bold')
ax_roc.legend(loc="lower right", fontsize=9)
ax_roc.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("七模型综合对比可视化完成。")

## 12. 真实裂缝图片测试

使用 `data/real_test/` 目录下的实际裂缝照片，对综合表现最优的模型进行推理测试。

> 注意：`data/real_test/` 目录用于存放实际工程场景拍摄的裂缝照片（非训练集来源），用于验证模型在真实场景下的泛化能力。如果该目录为空，本节将跳过实际推理并提示用户添加测试图片。

In [ ]:
# ========================================
# 12.1 真实裂缝图片推理测试
# ========================================

REAL_TEST_DIR = DATA_ROOT / "real_test"

# 选取最优模型（按F1分数）
best_model_name = df_all.iloc[0]["模型"]
best_factory = model_factories[best_model_name]
best_model = best_factory()
best_model.fit(X_train, y_train)
print(f"使用最优模型: {best_model_name}")

if not REAL_TEST_DIR.exists() or not REAL_TEST_DIR.is_dir():
    print(f"\n真实测试目录不存在: {REAL_TEST_DIR}")
    print("请将实际裂缝照片放入此目录后重试。")
elif not list(REAL_TEST_DIR.glob("*.jpg")) + list(REAL_TEST_DIR.glob("*.png")):
    print(f"\n真实测试目录为空: {REAL_TEST_DIR}")
    print("请将实际裂缝照片（.jpg/.png）放入此目录后重试。")
else:
    image_paths = sorted(
        [p for p in REAL_TEST_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS]
    )[:20]

    results_real = []
    n_cols = min(4, len(image_paths))
    n_rows = (len(image_paths) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
    if len(image_paths) == 1:
        axes = np.array([axes])
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else np.array([axes])

    for idx, img_path in enumerate(image_paths):
        img = _imread_gray(img_path)
        if img is None:
            continue

        processed = default_preprocess(img)
        features = extract_all_features(processed).reshape(1, -1)
        y_pred = best_model.predict(features)[0]
        try:
            proba = best_model.predict_proba(features)[0]
            confidence = proba.max()
        except Exception:
            confidence = float("nan")

        result_text = "有裂缝" if y_pred == 1 else "无裂缝"
        results_real.append({
            "文件名": img_path.name,
            "预测结果": result_text,
            "置信度": round(float(confidence), 4),
        })

        axes_flat[idx].imshow(img, cmap='gray')
        color = '#e74c3c' if y_pred == 1 else '#2ecc71'
        axes_flat[idx].set_title(
            f"{result_text}\n(置信度={confidence:.2f})",
            color=color, fontsize=10, fontweight='bold')
        axes_flat[idx].axis('off')

    # 隐藏多余子图
    for j in range(len(image_paths), len(axes_flat)):
        axes_flat[j].axis('off')

    plt.suptitle(f"真实裂缝图片测试结果 (模型: {best_model_name})",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    display(pd.DataFrame(results_real))
    print(f"\n共测试 {len(results_real)} 张真实图像。")

## 13. 本章小结

### 13.1 核心结论

1. **集成方法（RF/XGBoost/LightGBM）通常最优**：在裂缝识别任务中，基于树的集成方法（Bagging 和 Boosting）在高维特征空间表现优于单模型方法。

2. **朴素贝叶斯训练最快**：作为概率方法，训练速度远超其他模型，适合快速 baseline 或在线学习场景。

3. **SVM 在小样本下泛化好**：核函数提供非线性决策边界，但大数据集训练较慢。

4. **逻辑回归提供基准线**：线性模型简单高效，可解释性强，适合作为性能下限参考。

5. **决策树可解释性最强**：可视化决策规则有助于理解模型判断依据，是工业应用的重要考量。

### 13.2 推荐方案

| 场景 | 推荐模型 | 理由 |
|------|----------|------|
| 最高精度 | XGBoost / LightGBM | Boosting 精度最优 |
| 快速训练 | 朴素贝叶斯 | 训练极快 |
| 可解释性 | 决策树 | 可视化理解决策过程 |
| 平衡精度与速度 | 随机森林 | Bagging 抗过拟合，训练可并行 |

### 13.3 与 PDF 要求的对应

| PDF 要求 | 本章覆盖 |
|----------|:--:|
| 监督学习模型构建 | ✅ 7种模型全覆盖 |
| 不同模型对比 | ✅ 统一评估 + ROC叠加 |
| 参数调优 | ✅ GridSearchCV 搜索 |
| 模型验证 | ✅ 混淆矩阵 + ROC + CV |
| 实际图片测试 | ✅ real_test/ 推理 |

## 14. Gradio 接口预留

传统模型模块的 Gradio 回调接口。可视化系统中用户可选择模型类型、预处理方式和模型参数。

In [ ]:
# ========================================
# 14.1 传统模型 Gradio 回调接口（完整实现）
# ========================================

def traditional_model_handler(
    model_type: str = "random_forest",
    preprocessing: list = None,
    features: list = None,
    model_params: dict = None,
    cv_folds: int = 5,
    max_samples: int = 2000,
    random_seed: int = 42,
) -> dict:
    """Gradio 回调接口：传统机器学习模型训练与评估。

    优先从磁盘加载预训练模型；若无匹配模型则现场 GridSearchCV 训练。

    Returns
    -------
    dict: {model_name, metrics, confusion_matrix, roc_curve,
           precision_recall, feature_importance, prob_distribution,
           cv_results, best_params, model_path, training_time}
    """
    import time, json
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from sklearn.metrics import precision_recall_curve, average_precision_score

    if preprocessing is None:
        preprocessing = ["clahe", "median"]
    if features is None:
        features = ["hog", "lbp", "glcm", "edge_density"]

    MODEL_KEY_MAP = {
        "decision_tree": "decision_tree", "svm": "svm",
        "naive_bayes": "naive_bayes", "random_forest": "random_forest",
        "logistic_regression": "logistic_regression", "xgboost": "xgboost",
        "lightgbm": "lightgbm",
    }
    # Security: strict allowlist — reject unknown model types
    if model_type not in MODEL_KEY_MAP:
        raise ValueError(f"不支持的模型类型: {model_type}，可选: {list(MODEL_KEY_MAP.keys())}")
    model_key = MODEL_KEY_MAP[model_type]
    trad_dir = (Path(PROJECT_ROOT) / "outputs" / "models" / "traditional").resolve()
    model_path = trad_dir / f"{model_key}_best.joblib"
    # Security: guard against path traversal
    if model_path.resolve().parent != trad_dir:
        raise RuntimeError(f"模型路径越界: {model_path}")

    # Load data for evaluation
    n_per_class = max(100, max_samples // 4)
    images, labels = load_dataset(DATA_ROOT, max_samples=n_per_class)
    processed = np.array([default_preprocess(img) for img in images])
    X_val = np.stack([extract_all_features(img) for img in processed])
    y_val = labels
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_val, y_val, test_size=0.3, random_state=random_seed, stratify=y_val)

    # Try loading pretrained model
    t0 = time.time()
    if model_params is None and model_path.exists():
        print(f"加载预训练模型: {model_path}")
        best_model = joblib.load(model_path)
        params_path = trad_dir / f"{model_key}_best_params.json"
        if params_path.exists():
            with open(params_path) as f:
                model_params = json.load(f)
    else:
        # Train from scratch (simplified grid — see §4-§10 for full GridSearchCV)
        images_full, labels_full = load_dataset(DATA_ROOT, max_samples=max_samples // 2)
        processed_full = np.array([default_preprocess(img) for img in images_full])
        X_full = np.stack([extract_all_features(img) for img in processed_full])
        y_full = labels_full

        if model_type == "decision_tree":
            base = DecisionTreeClassifier(random_state=random_seed)
            gp = {'max_depth': [5, 10, 15, None], 'criterion': ['gini', 'entropy'], 'min_samples_split': [2, 5]}
        elif model_type == "svm":
            base = Pipeline([('s', StandardScaler()), ('m', SVC(probability=True, random_state=random_seed))])
            gp = {'m__kernel': ['linear', 'rbf'], 'm__C': [0.1, 1, 10]}
        elif model_type == "naive_bayes":
            base = Pipeline([('s', StandardScaler()), ('m', GaussianNB())])
            gp = {'m__var_smoothing': [1e-9, 1e-7, 1e-5]}
        elif model_type == "random_forest":
            base = RandomForestClassifier(random_state=random_seed, n_jobs=-1)
            gp = {'n_estimators': [50, 100, 200], 'max_depth': [10, 20, None], 'min_samples_split': [2, 5]}
        elif model_type == "logistic_regression":
            base = Pipeline([('s', StandardScaler()), ('m', LogisticRegression(random_state=random_seed, max_iter=2000))])
            gp = {'m__C': [0.1, 1, 10], 'm__penalty': ['l1', 'l2'], 'm__solver': ['liblinear']}
        elif model_type == "xgboost":
            from xgboost import XGBClassifier
            base = XGBClassifier(random_state=random_seed, n_jobs=-1, verbosity=0)
            gp = {'n_estimators': [50, 100], 'max_depth': [3, 6], 'learning_rate': [0.1, 0.3]}
        elif model_type == "lightgbm":
            from lightgbm import LGBMClassifier
            base = LGBMClassifier(random_state=random_seed, n_jobs=-1, verbose=-1)
            gp = {'n_estimators': [50, 100], 'max_depth': [3, 6], 'learning_rate': [0.1, 0.3]}
        grid = GridSearchCV(base, gp, cv=3, scoring='f1', n_jobs=-1, verbose=0)
        grid.fit(X_full, y_full)
        best_model = grid.best_estimator_
        if model_params is None:
            model_params = grid.best_params_

    train_time = time.time() - t0

    # Evaluate
    best_model.fit(X_tr, y_tr)
    y_pred = best_model.predict(X_te)
    try:
        y_prob = best_model.predict_proba(X_te)[:, 1]
        auc = roc_auc_score(y_te, y_prob)
    except Exception:
        y_prob = None; auc = float('nan')

    metrics = {
        "accuracy": float(accuracy_score(y_te, y_pred)),
        "precision": float(precision_score(y_te, y_pred, zero_division=0)),
        "recall": float(recall_score(y_te, y_pred, zero_division=0)),
        "f1": float(f1_score(y_te, y_pred, zero_division=0)),
        "roc_auc": float(auc),
    }

    # ---- 混淆矩阵 ----
    cm = confusion_matrix(y_te, y_pred, labels=[0, 1])
    fig_cm, ax_cm = plt.subplots(figsize=(5, 4))
    ax_cm.imshow(cm, cmap="Blues")
    ax_cm.set_xticks([0,1]); ax_cm.set_yticks([0,1])
    ax_cm.set_xticklabels(["预测 无裂缝", "预测 有裂缝"])
    ax_cm.set_yticklabels(["实际 无裂缝", "实际 有裂缝"])
    for i in range(2):
        for j in range(2):
            ax_cm.text(j, i, str(cm[i,j]), ha="center", va="center", fontsize=14,
                      fontweight="bold", color="white" if cm[i,j] > cm.max()/2 else "black")
    ax_cm.set_title(f"{model_type} - 混淆矩阵\n(F1={metrics['f1']:.4f})")
    plt.tight_layout()

    # ---- ROC 曲线 ----
    fig_roc = None
    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_te, y_prob)
        fig_roc, ax_roc = plt.subplots(figsize=(5, 4))
        ax_roc.plot(fpr, tpr, 'r-', linewidth=2, label=f"AUC={auc:.4f}")
        ax_roc.plot([0,1], [0,1], 'gray', linestyle='--', label="随机猜测")
        ax_roc.set_xlabel("假阳性率"); ax_roc.set_ylabel("真阳性率")
        ax_roc.set_title(f"{model_type} - ROC 曲线")
        ax_roc.legend(); ax_roc.grid(True, alpha=0.3)
        plt.tight_layout()

    # ==== NEW: Precision-Recall 曲线 ====
    fig_pr = None
    if y_prob is not None:
        precisions, recalls, thresholds = precision_recall_curve(y_te, y_prob)
        ap = average_precision_score(y_te, y_prob)
        fig_pr, ax_pr = plt.subplots(figsize=(5, 4))
        ax_pr.plot(recalls, precisions, 'b-', linewidth=2, label=f"AP={ap:.4f}")
        ax_pr.axhline(y=np.mean(y_te), color='gray', linestyle='--',
                      label=f"baseline (正类比例={np.mean(y_te):.2f})")
        ax_pr.set_xlabel("召回率"); ax_pr.set_ylabel("精确率")
        ax_pr.set_xlim([0.0, 1.0]); ax_pr.set_ylim([0.0, 1.05])
        ax_pr.set_title(f"{model_type} - Precision-Recall 曲线")
        ax_pr.legend(); ax_pr.grid(True, alpha=0.3)
        plt.tight_layout()

    # ==== NEW: 特征重要性（仅树模型）====
    fig_fi = None
    TREE_MODELS = {"decision_tree", "random_forest", "xgboost", "lightgbm"}
    if model_type in TREE_MODELS:
        # Extract feature importances from the estimator
        try:
            if hasattr(best_model, 'feature_importances_'):
                importances = best_model.feature_importances_
            elif hasattr(best_model, 'named_steps'):
                # Pipeline — try to get the inner estimator
                inner = best_model
                for step_name in ['m', 'logreg', 'svc', 'nb']:
                    if step_name in best_model.named_steps:
                        inner = best_model.named_steps[step_name]
                        break
                if hasattr(inner, 'feature_importances_'):
                    importances = inner.feature_importances_
                elif hasattr(inner, 'coef_'):
                    importances = np.abs(inner.coef_).flatten()
                else:
                    importances = None

            if importances is not None:
                # Use feature group names instead of 26352 individual dims
                n_total = len(importances)
                # Split into feature groups: HOG (~26244), LBP (59), GLCM (48), Edge (1)
                hog_end = 26244; lbp_end = hog_end + 59; glcm_end = lbp_end + 48
                group_importances = [
                    np.sum(importances[:hog_end]),
                    np.sum(importances[hog_end:lbp_end]) if n_total > hog_end else 0,
                    np.sum(importances[lbp_end:glcm_end]) if n_total > lbp_end else 0,
                    np.sum(importances[glcm_end:]) if n_total > glcm_end else 0,
                ]
                group_names = ["HOG", "LBP", "GLCM", "边缘密度"]
                # Normalize to sum=1
                total = np.sum(group_importances)
                if total > 0:
                    group_importances = [gi / total for gi in group_importances]

                fig_fi, ax_fi = plt.subplots(figsize=(7, 4))
                colors_fi = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
                bars = ax_fi.barh(group_names, group_importances, color=colors_fi)
                ax_fi.set_xlabel("特征组重要性 (归一化)")
                ax_fi.set_title(f"{model_type} - 特征重要性")
                for bar, val in zip(bars, group_importances):
                    ax_fi.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                              f'{val:.3f}', va='center', fontsize=10)
                ax_fi.set_xlim(0, max(group_importances) * 1.3)
                plt.tight_layout()
        except Exception:
            pass  # feature importance extraction failed — return None

    # ==== NEW: 预测概率分布 ====
    fig_prob = None
    if y_prob is not None:
        fig_prob, ax_prob = plt.subplots(figsize=(6, 4))
        pos_probs = y_prob[y_te == 1]
        neg_probs = y_prob[y_te == 0]
        bins = np.linspace(0, 1, 31)
        ax_prob.hist(neg_probs, bins=bins, alpha=0.6, color='#2ecc71',
                     label=f'实际无裂缝 (n={len(neg_probs)})', edgecolor='white')
        ax_prob.hist(pos_probs, bins=bins, alpha=0.6, color='#e74c3c',
                     label=f'实际有裂缝 (n={len(pos_probs)})', edgecolor='white')
        ax_prob.axvline(x=0.5, color='gray', linestyle='--', alpha=0.7, label='决策阈值(0.5)')
        ax_prob.set_xlabel("预测概率 (正类)"); ax_prob.set_ylabel("样本数")
        ax_prob.set_title(f"{model_type} - 预测概率分布")
        ax_prob.legend()
        plt.tight_layout()

    # Quick CV
    try:
        skf = StratifiedKFold(n_splits=min(3, cv_folds), shuffle=True, random_state=random_seed)
        fold_f1s = []
        for ti, vi in skf.split(X_val, y_val):
            m = clone(best_model)
            m.fit(X_val[ti], y_val[ti])
            fold_f1s.append(f1_score(y_val[vi], m.predict(X_val[vi]), zero_division=0))
        cv_means = {"f1_mean": float(np.mean(fold_f1s)), "f1_std": float(np.std(fold_f1s))}
    except Exception:
        cv_means = {"f1_mean": metrics["f1"], "f1_std": 0.0}

    return {
        "model_name": model_type,
        "metrics": metrics,
        "confusion_matrix": fig_cm,
        "roc_curve": fig_roc,
        "precision_recall": fig_pr,           # NEW
        "feature_importance": fig_fi,          # NEW (tree models only)
        "prob_distribution": fig_prob,         # NEW
        "cv_results": cv_means,
        "best_params": model_params,
        "model_path": str(model_path) if model_path.exists() else None,
        "training_time": round(train_time, 3),
    }


print("=" * 70)
print("Gradio 接口：traditional_model_handler — 已实现")
print("=" * 70)
print("输出图表: confusion_matrix, roc_curve, precision_recall,")
print("          feature_importance (树模型), prob_distribution")
print("=" * 70)